# 06 — Control 3: standardization

### How to read this notebook
Run cells top to bottom. Each Markdown cell states the question and expected result **before** the code. The printed and plotted numbers are measurements, not claims made in advance. The shared functions live in `src/experiment.py`. Open Jupyter from the project root so imports work.

## Why this control?

If $x$ is scaled by positive $\alpha$, its mean and standard deviation also scale: $\mu_\alpha=\alpha\mu$ and $\sigma_\alpha=\alpha\sigma$. Consequently $(\alpha x-\alpha\mu)/(\alpha\sigma)=(x-\mu)/\sigma$.

This control checks whether preprocessing removes the scaling intervention. It is **not** used in the main experiment.

In [1]:
from pathlib import Path
import os
import sys

# Resolve this experiment before imports and relative result paths.
ROOT = Path.cwd()
if not (ROOT / 'src' / 'experiment.py').is_file():
    ROOT = ROOT / 'experiment_1'
assert (ROOT / 'src' / 'experiment.py').is_file(), 'Start Jupyter in SimpBias or experiment_1.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment import *
import torch
import numpy as np
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)

PyTorch: 2.14.1+cpu


In [2]:
x1,y=make_data(1024,11,1)
for alpha in [0.1,1,5,10]:
    xa,_=make_data(1024,11,alpha)
    def standardize(x):
        return (x-x.mean(dim=0))/(x.std(dim=0,unbiased=False)+1e-12)
    assert torch.allclose(standardize(x1),standardize(xa),atol=1e-6,rtol=1e-6)
    difference=(standardize(x1)-standardize(xa)).abs().max().item()
    print(f'alpha={alpha}: max standardized-input difference={difference:.3g}')

alpha=0.1: max standardized-input difference=1.19e-07
alpha=1: max standardized-input difference=0
alpha=5: max standardized-input difference=1.19e-07
alpha=10: max standardized-input difference=1.19e-07


## Interpretation

After coordinate-wise standardization, the scaled inputs should agree with the reference to floating-point precision. If your main pipeline automatically standardizes each input coordinate, it can erase the intended treatment.